In [1]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')
print("Working directory:", os.getcwd())

Working directory: C:\Users\Klara\retail-intelligence


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

risk_table = pd.read_csv('data/processed/customer_risk_table.csv')

# Plot 1: Risk tier distribution
fig = px.pie(
    risk_table, 
    names='RiskTier',
    title='Customer Risk Tier Distribution',
    color='RiskTier',
    color_discrete_map={
    '⚫ Extreme Risk': '#4d0000',
    '🔴 High Risk': 'red',
    '🟡 Medium Risk': 'orange',
    '🟢 Low Risk': 'green'
}
)
fig.show()

In [3]:
# Plot 2: Revenue at risk by tier
tier_revenue = risk_table.groupby('RiskTier')['RevenueAtRisk'].sum().reset_index()
fig2 = px.bar(
    tier_revenue,
    x='RiskTier',
    y='RevenueAtRisk',
    title='Revenue at Risk by Customer Tier',
    labels={'RevenueAtRisk': 'Revenue at Risk ($)'},
    color='RiskTier',
    color_discrete_map={
    '⚫ Extreme Risk': '#4d0000',
    '🔴 High Risk': 'red',
    '🟡 Medium Risk': 'orange',
    '🟢 Low Risk': 'green'
}
    
    
)
fig2.show()

In [4]:
extreme = risk_table[risk_table['RiskTier'] == '⚫ Extreme Risk']
print("Extreme Risk — Frequency:")
print(extreme['Frequency'].describe())
print()
print("Extreme Risk — DaysActive:")
print(extreme['DaysActive'].describe())
print()
print("Extreme Risk — Monetary:")
print(extreme['Monetary'].describe())

Extreme Risk — Frequency:
count    75.000000
mean      1.186667
std       0.484722
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       3.000000
Name: Frequency, dtype: float64

Extreme Risk — DaysActive:
count     75.000000
mean       5.000000
std       17.764982
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      111.000000
Name: DaysActive, dtype: float64

Extreme Risk — Monetary:
count     75.000000
mean     165.870800
std      110.918902
min        6.900000
25%       99.820000
50%      127.100000
75%      206.400000
max      563.940000
Name: Monetary, dtype: float64


In [5]:
high = risk_table[risk_table['RiskTier'] == '🔴 High Risk']
print("High Risk — Frequency:")
print(high['Frequency'].describe())
print()
print("High Risk — DaysActive:")
print(high['DaysActive'].describe())
print()
print("High Risk — Monetary:")
print(high['Monetary'].describe())

High Risk — Frequency:
count    359.000000
mean       1.763231
std        1.004007
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        6.000000
Name: Frequency, dtype: float64

High Risk — DaysActive:
count    359.000000
mean      46.417827
std       66.356691
min        0.000000
25%        0.000000
50%        0.000000
75%       81.000000
max      265.000000
Name: DaysActive, dtype: float64

High Risk — Monetary:
count      359.000000
mean       667.243120
std       2373.338238
min         52.000000
25%        252.165000
50%        380.600000
75%        676.650000
max      44534.300000
Name: Monetary, dtype: float64


In [6]:
# Plot 3: Churn probability distribution
fig3 = px.histogram(
    risk_table,
    x='ChurnProbability',
    nbins=30,
    title='Distribution of Churn Probabilities',
    labels={'ChurnProbability': 'Churn Probability'}
)
fig3.show()

In [7]:
# Plot 4: Top churn drivers
driver_counts = risk_table['TopChurnDriver'].value_counts().reset_index()
driver_counts.columns = ['Feature', 'Count']
fig4 = px.bar(
    driver_counts,
    x='Feature',
    y='Count',
    title='Most Common Top Churn Driver per Customer'
)
fig4.show()

print("Sample of high risk customers:")
high_risk = risk_table[risk_table['RiskTier'] == '🔴 High Risk']
print(high_risk[['ChurnProbability', 'CLV', 'RevenueAtRisk', 
                  'TopChurnDriver', 'Explanation']].head(5))

Sample of high risk customers:
    ChurnProbability        CLV  RevenueAtRisk  TopChurnDriver  \
0           0.647206  53441.160   34587.452591      DaysActive   
7           0.596912   5177.664    3090.611453      DaysActive   
10          0.541450   4866.864    2635.163048  UniqueProducts   
18          0.563840   3831.048    2160.097769      DaysActive   
30          0.597031   2567.712    1533.002738      DaysActive   

                                          Explanation  
0   This customer has a high churn risk (64.7%). T...  
7   This customer has a high churn risk (59.7%). T...  
10  This customer has a high churn risk (54.1%). T...  
18  This customer has a high churn risk (56.4%). T...  
30  This customer has a high churn risk (59.7%). T...  


## Risk Segmentation — Observations


**1. Risk tier distribution (pie chart)**

45.8% High Risk (359), 23.1% Low Risk (181), 21.6% Medium Risk (169), 9.6% Extreme Risk (75). Splitting the original High Risk tier (55.4%) into High + Extreme reveals that the large majority of previously-"high risk" customers (0.5-0.7 probability) are meaningfully different from the small tail at 0.7+.

**2. Revenue at risk by tier (bar chart)**

Extreme Risk carries by far the *lowest* revenue at risk of any tier (~$11,183 across 75 customers, ~$149/customer) — less than Low Risk customers, despite being the group most confident to churn. High and Medium Risk carry the largest revenue exposure (~$175k and ~$120k), with higher per-customer averages (~$486 and ~$710) than Extreme Risk.

**3. Extreme Risk ≠ highest priority (confirmed)**

Checked directly: among the 75 Extreme Risk customers, `Frequency` has a median of 1 (75th percentile also 1) and `DaysActive` has a median of 0 (75th percentile also 0). This confirms the large majority of this tier are true one-time buyers — a single purchase, on a single day, with no repeat behavior at all. This is the same pattern first flagged in Day 2 (Customer 654: DaysActive=1, Frequency=1).

This explains why Extreme Risk carries the lowest revenue at risk of any tier despite the highest churn probability: `DaysActive` dominates the model's predictions (~85%+ of customers, per the driver chart), so customers with a same-day-only purchase history get pushed to very high probability almost automatically — regardless of how much they actually spent or whether they were ever a "regular" in the first place.

**Practical implication:** these customers never established a repeat relationship, so "winning them back" is a different (likely harder, lower-ROI) problem than re-engaging a lapsed regular. Retention spend is better directed at High and Medium Risk tiers, where churn probability is still meaningfully elevated but customers have demonstrated real repeat value.

**4. Extreme vs High Risk — direct comparison**

Compared Extreme Risk (n=75) against High Risk (n=359) across the same 
three metrics:

| Metric | Extreme Risk | High Risk |
|---|---|---|
| Frequency (median / 75th %ile / max) | 1 / 1 / 3 | 1 / 2 / 6 |
| DaysActive (median / 75th %ile / mean) | 0 / 0 / 5.0 | 0 / 81 / 46.4 |
| Monetary (median / 75th %ile / max) | $127 / $206 / $564 | $381 / $677 / $44,534 |

Both tiers share a median Frequency of 1 and median DaysActive of 0, so medians alone understate the difference between them — it shows up in the spread instead. High Risk's 75th percentile DaysActive (81 days) and mean (46.4) reveal a substantial share of genuinely tenured customers that Extreme Risk simply doesn't have  (Extreme Risk stays at 0 through its own 75th percentile). Monetary makes the gap unambiguous: High Risk's median spend ($381) is 3x Extreme Risk's ($127), and its maximum spender ($44,534) is roughly 80x Extreme Risk's maximum ($564) — High Risk contains real high-value customers; Extreme Risk does not.

**Conclusion:** Extreme Risk is a narrow, uniform group of low-spend, one-time buyers. High Risk is a broader, meaningfully more valuable population that includes customers with real tenure and spend. This reinforces the priority ordering above — High and Medium Risk are the tiers worth the retention investment; Extreme Risk mostly isn't.


**5. Churn probability distribution (histogram)**

The distribution is clearly bimodal, not smooth — a tall spike near 0.05-0.1 and an even taller spike around 0.65-0.7, with a comparatively sparse middle region (~0.1-0.5). This suggests the model is generally decisive rather than uncertain: most customers get pushed toward one of two clusters (clearly safe or clearly risky) rather than landing in an ambiguous middle ground. This likely connects to `DaysActive`'s outsized influence (see next point) — a feature that dominates predictions this strongly tends to produce sharp splits rather than a smooth probability gradient.

**6. Most common top churn driver per customer (bar chart)**

`DaysActive` is the single largest SHAP contributor for ~670-680 of 784 customers (~85%+), with `UniqueProducts` a distant second (~100 customers). This confirms and sharpens Day 1's global finding (DaysActive's mean |SHAP| was ~3x the next feature) — at the individual level, it's not just the most important feature on average, it's the deciding factor for the large majority of customers specifically. Worth being upfront about this in discussion: the model functions closer to DaysActive-driven, with six supporting features" than a balanced multi-feature model. Not necessarily a problem, but an honest characterization of how the model actually behaves.